# Logit Lens Evaluation Plots

This notebook generates plots for Logit Lens evaluation accuracy across multiple datasets and models.

## Setup

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.gridspec import GridSpec
from safetensors.numpy import load_file

# Set project root path (adjust if necessary)
PROJECT_ROOT = os.path.expanduser("~/scratch/plm")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots", "logitlens")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Reading results from: {RESULTS_DIR}")
print(f"Saving plots to: {OUTPUT_DIR}")

# Set seaborn style (Adhering to requested standard)
sns.set_style(
    "whitegrid",
    rc={"axes.edgecolor": ".15", "xtick.bottom": True, "ytick.left": True},
)


## Configuration

Define the datasets and models to include in the plots.

In [20]:
DATASETS = [
    ("l11p/COCOQA-restval-ICQA", ["images_qa"]),
    ("l11p/VQAv2-COCO-restval-ICQ", ["images_qa"]),
    ("lmms-lab/ScienceQA", ["images_qa"]),
    ("l11p/MME-test-captioned", ["images_qa"]),
    ("l11p/MMBench_EN-dev-captioned", ["images_qa"]),
    ("l11p/GQA-val-balanced-noyesno-captioned", ["images_qa"]),
    # ("lmms-lab/ai2d", ["images_qa"]),
    ("lmms-lab/POPE", ["images_qa"]),
    # ("lmms-lab/textvqa", ["images_qa"]),
    ("Lin-Chen/MMStar", ["images_qa"]),
    ("vikhyatk/CountBenchQA", ["images_qa"]),
]

# Multiple choice datasets for the comparison plot
MC_DATASETS = [
    # "lmms-lab/ScienceQA",
    "l11p/MME-test-captioned",
    "l11p/MMBench_EN-dev-captioned",
    # "lmms-lab/ai2d",
    "lmms-lab/POPE",
    "Lin-Chen/MMStar",
]

# (model_path, model_label)
MODELS = [
    # ("l11p/llava-v1.5-7b-pretrained-lbasile", "Pretrained"),
    ("nyu-visionx/cambrian-8b", "Finetuned"),
    ("lbasile/llava_7b", "Finetuned"),
    ("lbasile/llava_13b", "Finetuned"),
    ("lbasile/llava_13b", "Finetuned"),
    ("lmms-lab/LLaVA-OneVision-1.5-8B-Instruct", "Finetuned"),
    ("lmms-lab/LLaVA-OneVision-1.5-4B-Instruct", "Finetuned"),
    ("OpenGVLab/InternVL2-8B", "Finetuned"),
]

def load_accuracies(results_dir, model_slug, dataset_slug, subset, mode="logit_lens"):
    """
    mode: 'logit_lens' or 'linear_probe'
    """
    # Try the new filename format first
    filename = f"{model_slug}_{dataset_slug}_{subset}_{mode}.safetensors"
    path = os.path.join(results_dir, "logitlens_evaluation", filename)
    
    if os.path.exists(path):
        data = load_file(path)
        return data["accuracies"]
    
    # Fallback for older file naming convention (only for logit_lens)
    if mode == "logit_lens":
        filename_legacy = f"{model_slug}_{dataset_slug}_{subset}.safetensors"
        path_legacy = os.path.join(results_dir, "logitlens_evaluation", filename_legacy)
        if os.path.exists(path_legacy):
            return load_file(path_legacy)["accuracies"]
            
    print(f"Warning: File not found: {path}")
    return None


## Main Analysis Plots
The following cell generates the two primary plots for each model:
1. **Logit Lens Performance**: Across all configured datasets.
2. **Logit Lens vs Linear Probe**: Specifically for MC benchmarks (ScienceQA, MMBench, MME).

In [ ]:
# Generate plots for each model

# Baselines for normalization
BASELINES = {
    "scienceqa": 0.2,
    "mmbench_en_dev_captioned": 0.25,
    "mme_test_captioned": 0.5,
    "cocoqa_restval_icqa": 0.0,
    "vqav2_coco_restval_icq": 0.0,
    "gqa_val_balanced_noyesno_captioned": 0.0,
    "ai2d": 0.25,
    "pope": 0.5,
    "textvqa": 0.0,
    "mmstar": 0.25,
    "countbenchqa": 0.0,
}

# Flag to toggle standard deviation shading
SHOW_STD = True

def normalize_profile(arr, baseline):
    if arr is None: return None
    max_val = np.max(arr)
    if max_val <= baseline:
        denom = 1.0 # Avoid division by zero/weird plots if max <= baseline
    else:
        denom = max_val - baseline
    return (arr - baseline) / denom

for model_path, model_label in MODELS:
    model_slug = model_path.split('/')[-1]
    
    # --- Plot 1: Logit Lens Performance (All Datasets) ---
    fig_combined = plt.figure(figsize=(4.5, 4))
    gs_combined = GridSpec(1, 1)
    ax_combined = fig_combined.add_subplot(gs_combined[0])
    
    # Abstraction phase styling
    ax_combined.axvspan(
        6, 17, alpha=0.15, color="steelblue", zorder=0, label="abstraction phase"
    )

    has_data_p1 = False
    for dataset_name, subsets in DATASETS:
        dataset_slug = dataset_name.split('/')[-1].replace("-", "_").lower()
        dataset_display = dataset_name.split('/')[-1].replace("-restval-ICQA", "").replace("-COCO-restval-ICQ", "").replace("_EN", "")
        
        baseline = BASELINES.get(dataset_slug, 0.0)

        for subset in subsets:
            acc = load_accuracies(RESULTS_DIR, model_slug, dataset_slug, subset, mode="logit_lens")
            if acc is not None:
                has_data_p1 = True
                # Normalize wrt its own max
                acc_norm = normalize_profile(acc, baseline)
                
                sns.lineplot(
                    x=range(len(acc_norm)),
                    y=acc_norm,
                    label=dataset_display,
                    linewidth=1,
                    marker="o",
                    markersize=4,
                    ax=ax_combined
                )

    if has_data_p1:
        ax_combined.set_xticks(np.array(list(range(0, 33, 4))))
        ax_combined.set_ylim(-0.05, 1.05)
        ax_combined.set_ylabel("performance", fontsize=14)
        ax_combined.set_xlabel("layers", fontsize=12)
        # Legend inside to avoid narrowing the plot
        ax_combined.legend(fontsize=8, loc='best')
        ax_combined.set_title(f"Logit Lens: {model_label}", fontsize=15)
        ax_combined.grid(True, alpha=0.4)
        
        gs_combined.tight_layout(fig_combined)
        
        out_name = f"logitlens_all_normalized_{model_slug}.png"
        plt.savefig(os.path.join(OUTPUT_DIR, out_name), bbox_inches='tight', dpi=300)
        plt.show()
        print(f"Saved Plot 1: {out_name}")
    else:
        plt.close()
        print(f"No logit lens data found for {model_label}")

    # --- Plot 2: MC Benchmarks (Logit Lens vs Linear Probe) ---
    fig_combined = plt.figure(figsize=(4.5, 4))
    gs_combined = GridSpec(1, 1)
    ax_combined = fig_combined.add_subplot(gs_combined[0])
    
    ax_combined.axvspan(
        6, 17, alpha=0.15, color="steelblue", zorder=0, label="abstraction phase"
    )
    
    has_data_p2 = False
    colors = plt.cm.tab10(np.linspace(0, 1, len(MC_DATASETS)))
    
    for idx, dataset_name in enumerate(MC_DATASETS):
        # Find subsets for this dataset
        subsets = next((s for n, s in DATASETS if n == dataset_name), [])
        dataset_slug = dataset_name.split('/')[-1].replace("-", "_").lower()
        dataset_display = dataset_name.split('/')[-1].replace("_EN", "")
        color = colors[idx]
        
        # Get baseline for this dataset
        baseline = BASELINES.get(dataset_slug, 0.0)
        
        for subset in subsets:
            # Load Data
            acc_ll = load_accuracies(RESULTS_DIR, model_slug, dataset_slug, subset, mode="logit_lens")
            acc_lp = load_accuracies(RESULTS_DIR, model_slug, dataset_slug, subset, mode="linear_probe")
            
            # Plot Logit Lens
            if acc_ll is not None:
                has_data_p2 = True
                acc_ll_norm = normalize_profile(acc_ll, baseline)
                sns.lineplot(
                    x=range(len(acc_ll_norm)),
                    y=acc_ll_norm,
                    label=f"{dataset_display} (LL)",
                    color=color,
                    marker="o",
                    markersize=4,
                    linewidth=1,
                    ax=ax_combined
                )
            
            # Plot Linear Probe
            if acc_lp is not None:
                has_data_p2 = True
                acc_lp_norm = normalize_profile(acc_lp, baseline)
                sns.lineplot(
                    x=range(len(acc_lp_norm)),
                    y=acc_lp_norm,
                    label=f"{dataset_display} (LP)",
                    color=color,
                    linestyle="--",
                    linewidth=1,
                    marker="x", # Distinguish markers
                    markersize=4,
                    ax=ax_combined
                )

    if has_data_p2:
        ax_combined.set_xticks(np.array(list(range(0, 33, 4))))
        ax_combined.set_ylim(-0.05, 1.05)
        ax_combined.set_ylabel("performance", fontsize=14)
        ax_combined.set_xlabel("layers", fontsize=12)
        # Legend inside to avoid narrowing the plot
        ax_combined.legend(fontsize=8, loc='best')
        ax_combined.set_title(f"LL vs LP: {model_label}\n(0=Base, 1=Own Max)", fontsize=12)
        ax_combined.grid(True, alpha=0.4)
        
        gs_combined.tight_layout(fig_combined)
        
        out_name = f"logitlens_vs_probe_normalized_{model_slug}.png"
        plt.savefig(os.path.join(OUTPUT_DIR, out_name), bbox_inches='tight', dpi=300)
        plt.show()
        print(f"Saved Plot 2: {out_name}")
    else:
        plt.close()
        print(f"No MC benchmark data found for {model_label}")
        
    print("-" * 50)


In [ ]:
# --- Plot 3: Aggregated MME/MMBench vs QA ---

# Define groups based on the DATASETS list
GROUP_MC = ["l11p/MME-test-captioned", "l11p/MMBench_EN-dev-captioned", "lmms-lab/POPE", "Lin-Chen/MMStar"]
# GROUP_MC = ["l11p/MMBench_EN-dev-captioned", "lmms-lab/POPE"]
GROUP_QA = ["l11p/GQA-val-balanced-noyesno-captioned", "l11p/COCOQA-restval-ICQA", "l11p/VQAv2-COCO-restval-ICQ"]

# Helper to average arrays of potentially different lengths (clips to min len)
def average_arrays(arrays):
    if not arrays:
        return None, None
    min_len = min(len(arr) for arr in arrays)
    # Clip all to min_len
    clipped = [arr[:min_len] for arr in arrays]
    return np.mean(clipped, axis=0), np.std(clipped, axis=0)

for model_path, model_label in MODELS:
    model_slug = model_path.split('/')[-1]
    
    # Containers for normalized profiles
    mc_ll_profiles = [] # Logit Lens for MME & MMB
    mc_lp_profiles = [] # Linear Probe for MME & MMB
    qa_ll_profiles = [] # Logit Lens for VQAv2 & COCOQA

    print(f"Processing model: {model_label}")

    for dataset_name, subsets in DATASETS:
        dataset_slug = dataset_name.split('/')[-1].replace("-", "_").lower()
        baseline = BASELINES.get(dataset_slug, 0.0)
        
        # We assume 1 subset for simplicity or take the first valid one
        for subset in subsets:
            # Load Raw
            raw_ll = load_accuracies(RESULTS_DIR, model_slug, dataset_slug, subset, mode="logit_lens")
            raw_lp = load_accuracies(RESULTS_DIR, model_slug, dataset_slug, subset, mode="linear_probe")
            print(raw_lp)
            # Normalize (Independent normalization per dataset-method)
            norm_ll = normalize_profile(raw_ll, baseline)
            norm_lp = normalize_profile(raw_lp, baseline)

            # Add to respective groups
            if dataset_name in GROUP_MC:
                if norm_ll is not None: mc_ll_profiles.append(norm_ll)
                if norm_lp is not None: mc_lp_profiles.append(norm_lp)
            
            if dataset_name in GROUP_QA:
                if norm_ll is not None: qa_ll_profiles.append(norm_ll)
                # We do not aggregate Probe for QA group as per instructions

    # Compute Means and STDs
    mean_mc_ll, std_mc_ll = average_arrays(mc_ll_profiles)
    mean_mc_lp, std_mc_lp = average_arrays(mc_lp_profiles)
    mean_qa_ll, std_qa_ll = average_arrays(qa_ll_profiles)
    print(mc_lp_profiles)
    # Plotting
    fig_combined = plt.figure(figsize=(4.5, 4))
    gs_combined = GridSpec(1, 1)
    ax_combined = fig_combined.add_subplot(gs_combined[0])
    
    # Abstraction phase
    ax_combined.axvspan(
        5.5, 16.5, alpha=0.25, color="goldenrod", zorder=0, label="abstraction phase"
    )
      # Abstraction phase
    ax_combined.axvspan(
        16.5,32, alpha=0.15, color="steelblue", zorder=0, label="generation phase"
    )

    if mean_mc_ll is not None:
        sns.lineplot(
            x=range(len(mean_mc_ll))[1::2], y=mean_mc_ll[1::2], 
            label="multiple choice (LL)", color="darkslategray",
            marker="o", markersize=4, linewidth=1, ax=ax_combined
        )
        if SHOW_STD and std_mc_ll is not None:
            ax_combined.fill_between(range(len(mean_mc_ll))[1::2], mean_mc_ll[1::2] - std_mc_ll[1::2]/2, mean_mc_ll[1::2] + std_mc_ll[1::2]/2, color="tab:blue", alpha=0.2)
    
    if mean_mc_lp is not None:
        sns.lineplot(
            x=range(len(mean_mc_lp))[1::2], y=mean_mc_lp[1::2],
            label="multiple choice (LP)", color="darkslategray", linestyle="--",
            marker="s", markersize=4, linewidth=1, ax=ax_combined
        )
        if SHOW_STD and std_mc_lp is not None:
            ax_combined.fill_between(range(len(mean_mc_lp))[1::2], mean_mc_lp[1::2] - std_mc_lp[1::2]/2, mean_mc_lp[1::2] + std_mc_lp[1::2]/2, color="tab:blue", alpha=0.1)
        
    if mean_qa_ll is not None:
        sns.lineplot(
            x=range(len(mean_qa_ll))[1::2], y=mean_qa_ll[1::2],
            label="open ended", color="darkred",
            marker="s", markersize=4, linewidth=1, ax=ax_combined
        )
        if SHOW_STD and std_qa_ll is not None:
            ax_combined.fill_between(range(len(mean_qa_ll))[1::2], mean_qa_ll[1::2] - std_qa_ll[1::2]/2, mean_qa_ll[1::2] + std_qa_ll[1::2]/2, color="tab:red", alpha=0.2)

    ax_combined.set_xticks(np.array(list(range(0, 33, 4))))
    ax_combined.set_ylim(-0.05, 1.05)
    ax_combined.set_ylabel("performance", fontsize=14)
    ax_combined.set_xlabel("layers", fontsize=12)
    
    # Place legend below the plot, centered
    ax_combined.legend(fontsize=10, loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2)
    
    # ax_combined.set_title(f"Aggregated: {model_label}", fontsize=15)
    ax_combined.grid(True, alpha=0.4)
    
    gs_combined.tight_layout(fig_combined)
    
    out_name = f"aggregated_normalized_{model_slug}.pdf"
    plt.savefig(os.path.join(OUTPUT_DIR, out_name), bbox_inches='tight')
    plt.show()
    print(f"Saved Aggregated Plot: {out_name}")
    print("-" * 50)
